# Discriminant Classifiers

Using discriminant classifiers / Naive Bayes to classify our data.

In [33]:
import pandas as pd

In [34]:
learn_data = pd.read_csv("scaled_train_fs.csv", header = None)
learn_data.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female', 'Target']
learn_data["Female"] = learn_data["Female"].astype("category")
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female,Target
0,0.173844,1.109406,0.362037,0.487504,-0.903219,-1.399329,1.203258,0,0
1,-0.379308,0.226878,-0.573270,0.288250,1.482342,1.488320,0.927290,0,0
2,-1.362690,-0.430090,-0.232378,0.575302,-0.024328,0.212382,-0.353374,0,0
3,-0.194924,-0.795164,-0.925510,0.589292,0.101228,0.413846,-0.458710,1,0
4,0.542612,2.761282,1.783803,-0.293097,0.352339,-0.459164,1.153956,1,0


In [35]:
learn_data.isna().value_counts()

Age    TB     Alkphos  Sgot   ALB    AR     BilRatio  Female  Target
False  False  False    False  False  False  False     False   False     449
Name: count, dtype: int64

In [36]:
X = learn_data.drop(columns = ["Target"])
y = learn_data["Target"]

## Metrics

In [79]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

## Linear Discriminant Classifier

From PCA analysis, we know that our data can be separated in two clouds of sick and healthy patients respectively. A LD classifier might work well, but we know that the clouds may overlap. Also, their covariance matrices are clearly different. We might need to use a Quadratic Discriminant Classifier instead.

In [38]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.33, random_state = 42)

lda_model = LinearDiscriminantAnalysis(priors = (0.5, 0.5))
lda_model.fit(X_train, y_train)

print('Priors:', lda_model.priors_)

Priors: [0.5 0.5]


In [39]:
confusion(np.array(y_train), pd.Series(lda_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	72	12
	0	82	134
Accuracy: 68.67%


In [40]:
confusion(np.array(y_val), pd.Series(lda_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	31	11
	0	37	70
Accuracy: 67.79%


In [41]:
from sklearn.model_selection import cross_validate

lda_model = LinearDiscriminantAnalysis(priors = (0.5, 0.5))
cross_val_results = pd.DataFrame(cross_validate(lda_model, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["LDA", :] = mean_results
metrics_df

,F1 Macro,Recall,Precision,Accuracy
LDA,0.648262,0.714125,0.674174,0.661648


## Quadratic Discrimant Classifier

We now use a QDA classifier. We see that the problem is preserved: the "sick" class overlaps too much with the healthy class and it

In [42]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

qda_model = QuadraticDiscriminantAnalysis(priors = (0.5, 0.5))
_ = qda_model.fit(X_train, y_train)

In [43]:
confusion(np.array(y_train), pd.Series(qda_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	73	11
	0	87	129
Accuracy: 67.33%


In [44]:
confusion(np.array(y_val), pd.Series(qda_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	34	8
	0	43	64
Accuracy: 65.77%


In [45]:
qda_model = QuadraticDiscriminantAnalysis(priors = (0.5, 0.5))
cross_val_results = pd.DataFrame(cross_validate(qda_model, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["QDA", :] = mean_results
metrics_df

,F1 Macro,Recall,Precision,Accuracy
LDA,0.648262,0.714125,0.674174,0.661648
QDA,0.627143,0.701966,0.665863,0.637054


QDA can be regularized with a parameter between 0 and 1, so we can apply cross-validation in an attempt to obtain better metrics. In general, a small value of this regularization parameter (between 0.01 and 0.1) is desirable, but it does not improve results by much.

In [46]:
cross_val_results = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

regs = np.logspace(start = -4, stop = -0.5, num = 100)

for reg in regs:
    qda_model = QuadraticDiscriminantAnalysis(priors = (0.5, 0.5), reg_param = reg)
    this_results = pd.DataFrame(cross_validate(qda_model, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
    mean_results = this_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
    cross_val_results.loc[reg, :] = mean_results

cross_val_results.sort_values(by = "F1 Macro", ascending = False).head()

,F1 Macro,Recall,Precision,Accuracy
0.016876,0.627691,0.704428,0.668245,0.637054
0.015557,0.627691,0.704428,0.668245,0.637054
0.014341,0.627691,0.704428,0.668245,0.637054
0.013219,0.627691,0.704428,0.668245,0.637054
0.012186,0.627691,0.704428,0.668245,0.637054


## Naive Bayes



In [47]:
from sklearn.naive_bayes import GaussianNB

gaussian_nb = GaussianNB(priors = (0.5, 0.5))
gaussian_nb.fit(X_train, y_train)

confusion(np.array(y_train), pd.Series(gaussian_nb.predict(X_train)))

		Predicted
		+1	0
Real	+1	72	12
	0	93	123
Accuracy: 65.00%


In [48]:
confusion(np.array(y_val), pd.Series(gaussian_nb.predict(X_val)))

		Predicted
		+1	0
Real	+1	33	9
	0	46	61
Accuracy: 63.09%


In [49]:
gaussian_nb = GaussianNB(priors = (0.5, 0.5))
cross_val_results = pd.DataFrame(cross_validate(gaussian_nb, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["GaussianNB", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LDA,0.648262,0.714125,0.674174,0.661648
GaussianNB,0.6398,0.713779,0.674974,0.650412
QDA,0.627143,0.701966,0.665863,0.637054


In [ ]:
gaussian_nb.fit(X, y)


## Logistic Regression

In [55]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

logreg_model = LogisticRegression(C = 20, random_state = 42, class_weight = "balanced")

logreg_model.fit(X_train, y_train)
confusion(np.array(y_train), pd.Series(logreg_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	72	12
	0	75	141
Accuracy: 71.00%


In [56]:
confusion(np.array(y_val), pd.Series(logreg_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	31	11
	0	34	73
Accuracy: 69.80%


In [59]:
Cs = np.logspace(start = -4, stop = 2, num = 100)

logreg_model = LogisticRegressionCV(Cs = Cs, random_state = 42, cv = 5, class_weight = "balanced")

logreg_model.fit(X, y)
confusion(np.array(y), pd.Series(logreg_model.predict(X)))

		Predicted
		+1	0
Real	+1	105	21
	0	125	198
Accuracy: 67.48%


In [60]:
avg_crossval_scores = logreg_model.scores_[1].mean(axis=0)
idx = np.argmax(avg_crossval_scores)
best_C = logreg_model.Cs_[idx]
print(best_C)

0.7564633275546291


In [61]:
logreg_model_best = LogisticRegression(C = best_C, random_state = 42)
cross_val_results = pd.DataFrame(cross_validate(logreg_model_best, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values
metrics_df.loc["LogReg-Best", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LDA,0.648262,0.714125,0.674174,0.661648
GaussianNB,0.6398,0.713779,0.674974,0.650412
QDA,0.627143,0.701966,0.665863,0.637054
LogReg-Best,0.538077,0.549298,0.588439,0.70387


## Trying our best classifiers on our test data

In [62]:
test_data = pd.read_csv("scaled_test_fs.csv", header = None)
test_data.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female']
test_data["Female"] = learn_data["Female"].astype("category")
test_data.head()

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female
0,-2.100226,-0.795164,1.907027,-0.567456,1.356786,1.555475,-1.512071,0
1,1.034303,0.171538,-0.117670,1.320148,1.105674,-0.459164,1.121330,0
2,0.911380,-0.795164,-0.643898,-1.387576,1.356786,0.548155,-0.458710,0
3,0.911380,1.351361,-0.212816,3.236675,0.101228,-0.526319,1.056650,1
4,0.173844,-0.537931,-0.631959,0.132669,-0.526551,-0.123391,-0.926871,1


In [71]:
test_y = pd.read_csv("test_y.csv").iloc[:, 1]
test_y

0      1
1      0
2      1
3      0
4      1
      ..
111    1
112    0
113    1
114    0
115    0
Name: Label, Length: 116, dtype: int64

### QDA

In [77]:
qda_model_best = QuadraticDiscriminantAnalysis(priors = (0.5, 0.5), reg_param = 0.01)
qda_model_best.fit(X, y)

labels_qda = pd.DataFrame(columns = ['ID', 'Label'])
labels_qda['Label'] = pd.DataFrame(qda_model_best.predict(test_data))
labels_qda['ID'] = labels_qda.index + 1
labels_qda

,ID,Label
0,1,0
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,0
113,114,1
114,115,0


In [89]:
confusion(test_y, labels_qda['Label'])

		Predicted
		+1	0
Real	+1	26	7
	0	29	54
Accuracy: 68.97%


In [92]:
compute_metrics(test_y, labels_qda['Label'])

[0.6704545454545454, 0.719240598758671, 0.6789865871833085, 0.6896551724137931]

### Naive Bayes

In [82]:
gaussian_nb = GaussianNB(priors = (0.5, 0.5))
gaussian_nb.fit(X, y)

labels_nb = pd.DataFrame(columns = ['ID', 'Label'])
labels_nb['Label'] = pd.DataFrame(gaussian_nb.predict(test_data))
labels_nb['ID'] = labels_nb.index + 1
labels_nb

,ID,Label
0,1,0
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,0
113,114,1
114,115,0


In [90]:
confusion(test_y, labels_nb['Label'])

		Predicted
		+1	0
Real	+1	24	9
	0	31	52
Accuracy: 65.52%


In [85]:
compute_metrics(labels_nb['Label'], test_y)

[0.6338383838383839,
 0.6444113263785395,
 0.6768893756845564,
 0.6551724137931034]

In [64]:
labels_qda.to_csv('new_predictions/qda_best_fs_nooversampling.csv', index = False)

In [86]:
randomforest_labels = pd.read_csv("random_forest_fs.csv")
randomforest_labels.head()

,ID,Label
0,1,0
1,2,0
2,3,1
3,4,0
4,5,0


In [88]:
confusion(test_y, randomforest_labels['Label'])

		Predicted
		+1	0
Real	+1	16	17
	0	16	67
Accuracy: 71.55%


In [93]:
compute_metrics(test_y, randomforest_labels['Label'])

[0.6473514509442653,
 0.6460387002555678,
 0.6488095238095238,
 0.7155172413793104]